# 05 — Deep Fine-Tuning  (D4 Levels 3 & 4)

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

The top of the fine-tuning ladder.

| Level | Technique | Applies to |
|---|---|---|
| **3a** | staged / warm-start boosting (`init_model`) | tabular: M1, M2, M3, M4, M7, M10 |
| **3b** | self-supervised FT-Transformer + **LoRA adapter per module** | all modules |
| **3c** | layer-wise unfreezing + discriminative learning rates | M9 sequence net |
| **4** | **EXTRA FINE-TUNING hook** — new campaigns from YAML, zero code changes | all |

### Level 3 is not "more Level 2"

Level 2 searches *hyper-parameters* and refits from scratch at every trial.
Level 3 **adapts a model that has already been fitted** — it continues training,
or transfers a learned representation to a new task. That distinction is the
whole point of the level.

### The honest note on PEFT/LoRA

The brief names *"PEFT/LoRA for transformers"*. **There is no text or image data
in this package** — all eight files are tabular or panel numeric. Adapting a
pretrained language model here would be decorative: there would be nothing for
its pretrained representations to transfer.

So LoRA is applied where it *is* meaningful. An FT-Transformer is pre-trained
**self-supervised** on the pooled tabular data (masked-feature reconstruction,
no labels), then each module attaches its own low-rank adapter and head while
the shared trunk stays frozen. That is exactly the PEFT pattern — one backbone,
many cheap task adapters — applied to the data that actually exists.

> Prerequisites: notebooks `00`–`04`.

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/01 ML in Finance/data"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from novafin.config import load_config
from novafin.data import load_all, make_feature_frame, make_splitter
from novafin.evaluate import average_precision, ks_statistic, roc_auc
from novafin.features import build_features
from novafin.models import load_model_specs
from novafin.models.campaign import list_campaigns, load_campaign, run_campaign, validate_campaign
from novafin.models.finetune import (
    FTTransformerConfig, LoRAConfig, attach_lora, build_ft_transformer,
    default_schedule, discriminative_learning_rates, layerwise_unfreeze_schedule,
    lora_parameter_report, pretrain_masked_features, staged_boosting,
)
from novafin.tracking import ExperimentTracker
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme, color, save_figure, semantic_color

cfg = load_config()
setup_logging("WARNING", log_file=cfg.paths.logs / "05_finetune.log")
seed_everything(cfg.reproducibility.seed)
apply_theme()
pd.set_option("display.width", 200)

tracker = ExperimentTracker(cfg)
print("config fingerprint:", cfg.fingerprint())
print("GPU enabled       :", cfg.compute.gpu_enabled())

try:
    import torch
    print("torch             :", torch.__version__, "| CUDA:", torch.cuda.is_available())
except ImportError:
    print("torch             : not installed - sections 3 and 4 will be skipped")

In [ ]:
# Rebuild the feature matrices (self-contained, matches the current fingerprint).
results = load_all(cfg=cfg)
features, matrices = {}, {}
for key, loaded in results.items():
    built = build_features(key, loaded.frame, cfg)
    features[key] = built
    spec = cfg.dataset(key)
    extra = [c for c in ("fwd_return_5d", "fwd_inflows_5d") if c in built.frame.columns]
    X, y = make_feature_frame(built.frame, spec, extra_drop=extra)
    X = X.drop(columns=[c for c in X.columns
                        if pd.api.types.is_datetime64_any_dtype(X[c])], errors="ignore")
    matrices[key] = (X, built.frame[built.target])

pd.DataFrame([{"module": k, "rows": len(X), "features": X.shape[1]}
              for k, (X, y) in matrices.items()])

## 2 · Level 3a — staged / warm-start boosting

### The mechanism, and why it is not just raising `n_estimators`

LightGBM's `init_model` **continues an existing booster** instead of starting a
new one. That lets the *learning schedule itself* change between stages, which a
single fit cannot do:

| Stage | Learning rate | Capacity | Regularisation | Role |
|---|---|---|---|---|
| 1 | 0.10 | 15 leaves | λ=0.1 | learn coarse structure fast |
| 2 | 0.03 | 31 leaves | λ=1.0 | refine stage 1's residual |
| 3 | 0.01 | 63 leaves | λ=5.0 | polish without memorising |

This is the gradient-boosting analogue of a learning-rate schedule in deep
learning, and it is the correct Level-3 technique for tabular data: it adapts an
**already fitted** model rather than refitting.

In [ ]:
X_loans, y_loans = matrices["loans"]
splitter_loans, kwargs_loans = make_splitter("loans", features["loans"].frame, cfg=cfg)

schedule = default_schedule(n_stages=4, trees_per_stage=200)
print("Staged schedule:")
display(pd.DataFrame(schedule))

staged = staged_boosting(
    X_loans, y_loans, splitter_loans,
    schedule=schedule,
    base_params={"min_data_in_leaf": 40, "feature_fraction": 0.8},
    scorer=ks_statistic, module="loans", cfg=cfg,
    split_kwargs=kwargs_loans, objective_name="ks",
    task="binary_classification", patience=2,
)

display(staged.to_frame())
print(f"\nstage 1 (coarse only) : {staged.baseline_score:.5f}")
print(f"best stage            : {staged.best_stage}  ->  {staged.best_score:.5f}")
print(f"improvement           : {staged.improvement:+.5f}")
for note in staged.notes:
    print("note:", note)

In [ ]:
# Does each stage earn its trees? Score against the cumulative tree count.
trace = staged.to_frame()
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(trace["trees_total"], trace["score"], marker="o", lw=2, color=color("teal"))
for _, row in trace.iterrows():
    ax.annotate(f"lr={row.get('param.learning_rate', '')}",
                (row["trees_total"], row["score"]),
                textcoords="offset points", xytext=(0, 10), fontsize=8, ha="center")
ax.axhline(staged.baseline_score, ls="--", lw=1.2, color=semantic_color("critical"),
           label=f"stage 1 = {staged.baseline_score:.4f}")
ax.set_xlabel("Cumulative trees"); ax.set_ylabel("Out-of-fold KS")
ax.set_title("M2 — staged boosting: each stage continues the previous booster")
ax.legend()
save_figure(fig, "05_m02_staged_boosting", close=False); plt.show()

print("A FLAT or FALLING line after stage 2 is a real and reportable result:")
print("it means the coarse stage already captured the structure, and the extra")
print("capacity is fitting noise. Early stopping on patience makes that visible.")

In [ ]:
# Compare against the Level-1 and Level-2 results for the same module.
summary_rows = [{"level": "L3 staged boosting", "ks": staged.best_score,
                 "detail": f"stage {staged.best_stage}, {trace['trees_total'].iloc[staged.best_stage-1]} trees"}]

baseline_path = cfg.paths.tables / "03_baseline_summary.csv"
if baseline_path.exists():
    base = pd.read_csv(baseline_path)
    match = base[(base["module"] == "loans") & (base["model"] == "lightgbm")]
    if len(match) and "ks" in match.columns:
        summary_rows.insert(0, {"level": "L1 baseline", "ks": float(match["ks"].iloc[0]),
                                "detail": "configs/models.yaml defaults"})

tuning_path = cfg.paths.tables / "04_tuning_summary.csv"
if tuning_path.exists():
    tuned = pd.read_csv(tuning_path)
    match = tuned[(tuned["module"] == "loans") & (tuned["model"] == "lightgbm")]
    if len(match):
        summary_rows.insert(-1, {"level": "L2 Optuna", "ks": float(match["tuned"].iloc[0]),
                                 "detail": f"{int(match['n_trials'].iloc[0])} trials"})

ladder = pd.DataFrame(summary_rows)
display(ladder.round(5))
print("\nThis three-row table IS the fine-tuning ladder for M2. Report it as is -")
print("including the case where a higher level does NOT beat a lower one.")

## 3 · Level 3b — self-supervised pre-training + LoRA

### Step 1: pool every module's features and pre-train with no labels

The masked-feature objective (zero 15% of each row, reconstruct it) is the
tabular analogue of masked language modelling. It learns the **joint structure
of the features**, which is shared across modules, rather than any one module's
target.

This step is what makes the LoRA story coherent: without a pretrained backbone
there is nothing to adapt, and "LoRA" would just be a randomly-initialised small
model wearing a PEFT label.

In [ ]:
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed - skipping sections 3 and 4.")

if TORCH_AVAILABLE:
    # Pool the numeric features of every module into one unlabelled corpus.
    # Columns differ per module, so each is standardised and padded to a common
    # width - the tokeniser learns a per-feature embedding either way.
    pooled_blocks, widths = [], []
    for key, (X, _) in matrices.items():
        numeric = X.select_dtypes(include=[np.number])
        numeric = numeric.fillna(numeric.median(numeric_only=True))
        standardised = (numeric - numeric.mean()) / (numeric.std() + 1e-9)
        pooled_blocks.append(standardised.to_numpy(dtype="float32"))
        widths.append(standardised.shape[1])

    width = int(np.median(widths))
    aligned = []
    for block in pooled_blocks:
        if block.shape[1] >= width:
            aligned.append(block[:, :width])
        else:
            pad = np.zeros((len(block), width - block.shape[1]), dtype="float32")
            aligned.append(np.hstack([block, pad]))

    pooled = np.vstack(aligned)
    pooled = np.nan_to_num(pooled, nan=0.0, posinf=0.0, neginf=0.0)
    print(f"pooled unlabelled corpus: {pooled.shape[0]:,} rows x {pooled.shape[1]} features")
    print(f"(contributions: {dict(zip(matrices.keys(), [len(b) for b in aligned]))})")

In [ ]:
if TORCH_AVAILABLE:
    backbone_cfg = FTTransformerConfig(n_features=pooled.shape[1], d_token=192,
                                       n_blocks=3, n_heads=8, dropout=0.1)
    backbone = build_ft_transformer(backbone_cfg)
    total_params = sum(p.numel() for p in backbone.parameters())
    print(f"FT-Transformer: {total_params:,} parameters "
          f"(d_token={backbone_cfg.d_token}, {backbone_cfg.n_blocks} blocks)")

    history = pretrain_masked_features(
        backbone, pooled, epochs=8, batch_size=512, mask_ratio=0.15, lr=1e-3,
        seed=cfg.reproducibility.seed,
    )
    display(history.round(5))

    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(history["epoch"], history["masked_mse"], marker="o", color=color("teal"))
    ax.set_xlabel("Epoch"); ax.set_ylabel("Masked reconstruction MSE")
    ax.set_title("Self-supervised pre-training (no labels used)")
    save_figure(fig, "05_pretraining_loss", close=False); plt.show()

    fall = 1 - history["masked_mse"].iloc[-1] / history["masked_mse"].iloc[0]
    print(f"reconstruction loss fell {fall:.1%} - the backbone learned feature structure")

### Step 2: attach a LoRA adapter per module

The trunk is **frozen**. Each module trains only its adapter (`A`, `B`) and its
own head.

$$W' = W + \frac{\alpha}{r} BA, \quad A \in \mathbb{R}^{r \times d}, \; B \in \mathbb{R}^{d \times r}$$

`B` is initialised to **zero**, so the adapter is an exact identity at step 0 —
fine-tuning can never begin from a worse point than the backbone.

In [ ]:
if TORCH_AVAILABLE:
    import copy

    rows = []
    for rank in (2, 4, 8, 16):
        adapted = attach_lora(copy.deepcopy(backbone), LoRAConfig(r=rank))
        report = lora_parameter_report(adapted)
        rows.append({
            "rank_r": rank,
            "total_parameters": report.attrs["total_parameters"],
            "trainable": report.attrs["trainable_parameters"],
            "trainable_pct": round(report.attrs["trainable_pct"], 3),
        })

    lora_table = pd.DataFrame(rows)
    display(lora_table)
    print("\nTHIS TABLE IS THE PEFT EVIDENCE. A module with 89 positives (churn)")
    print("cannot fit a full transformer without memorising - but it can plausibly")
    print("fit a few thousand adapter weights on top of a representation learned")
    print("from ~190,000 unlabelled rows.")

In [ ]:
if TORCH_AVAILABLE:
    # Which tensors remain trainable? Everything must be an adapter or a head.
    adapted = attach_lora(copy.deepcopy(backbone), LoRAConfig(r=8))
    report = lora_parameter_report(adapted)
    trainable = report[report["trainable"]]
    display(trainable.head(12))
    print(f"\ntrainable tensors: {len(trainable)} of {len(report)}")
    unexpected = [n for n in trainable["parameter"]
                  if "lora_" not in n and not n.startswith("head")]
    print("unexpected trainable tensors:", unexpected or "none")
    assert not unexpected, "something outside the adapters and head is trainable"

## 4 · Level 3c — layer-wise unfreezing and discriminative learning rates

Two techniques from ULMFiT (Howard & Ruder, 2018), applied to the M9 sequence
net.

**Gradual unfreezing.** A randomly-initialised head produces large, noisy
gradients in its first epochs. If the whole backbone is trainable, those
gradients destroy the pretrained representation — *catastrophic forgetting*.
Training the head alone first lets it reach a sensible region before any
backbone weight moves. Layers are then released **top-down**, because later
layers encode the most task-specific structure.

**Discriminative rates.** $\eta_{l-1} = \eta_l / 2.6$ — the head adapts fast,
the tokeniser barely moves.

In [ ]:
schedule_df = pd.DataFrame(layerwise_unfreeze_schedule(n_layers=3, epochs=8, warmup_epochs=2))
display(schedule_df)

rates = discriminative_learning_rates(n_layers=3, base_lr=1e-3, decay=2.6)
rates_df = pd.DataFrame([{"group": k, "learning_rate": v} for k, v in rates.items()])
display(rates_df)
print(f"the head trains {rates['head'] / rates['tokenizer']:,.0f}x faster than the tokeniser")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].step(schedule_df["epoch"], schedule_df["trainable_layers"].apply(len),
             where="post", lw=2, color=color("teal"))
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Trainable backbone layers")
axes[0].set_title("Gradual unfreezing (top-down)")
axes[0].set_yticks(range(0, 4))

axes[1].barh(rates_df["group"], rates_df["learning_rate"], color=color("navy"))
axes[1].set_xscale("log"); axes[1].set_xlabel("Learning rate (log scale)")
axes[1].set_title("Discriminative learning rates")
plt.tight_layout()
save_figure(fig, "05_unfreezing_and_lrs", close=False); plt.show()

## 5 · Level 4 — the EXTRA FINE-TUNING hook

> **The contract:** launch a new campaign — new search space, more trials,
> ensembling/stacking, threshold optimisation — **by editing YAML only, with
> zero code changes.**

### Why that is achievable

Each earlier phase removed one hard-coded thing:

* **Phase 4** — estimators became a dotted path in `configs/models.yaml`,
  resolved with `importlib`;
* **Phase 5** — search spaces became YAML, and a campaign may point at **its
  own** spaces file;
* **Phase 4** also exposed `pipeline_factory` and `fold_callback` on the CV loop,
  so a campaign can substitute a stacked or calibrated pipeline.

Level 4 is the payoff for those decisions, not a new mechanism.

In [ ]:
print("campaigns available:")
for path in list_campaigns():
    print("   ", path.name)

spec = load_campaign("configs/campaigns/example_extra_tuning.yaml")
print(f"\ncampaign  : {spec.name}")
print(f"author    : {spec.author}")
print(f"steps     : {len(spec.steps)} declared, {len(spec.enabled_steps())} enabled")
print(f"\n{' '.join(spec.description.split())}\n")
display(spec.plan())

problems = validate_campaign(spec)
print("validation:", "OK" if not problems else problems)

### 5.1 · The worked example

The scenario: the Level-2 studies are done and three follow-up questions arrive
— the kind that actually arrive after a first round of results.

1. *"The credit model looks close. What if you search harder?"* → **more trials**
2. *"Would combining the models beat any single one?"* → **stacking**
3. *"Our fraud team says a missed fraud really costs ₹25,000."* → **threshold**

Answering all three requires **zero changes to any file in `src/`**.

In [ ]:
# Dry run first: validates every step WITHOUT fitting anything.
# A campaign may run unattended for an hour, so a typo in step 4 must fail
# before step 1 starts.
dry = run_campaign("configs/campaigns/example_extra_tuning.yaml", cfg=cfg, dry_run=True)
print("dry run ok:", dry.ok)
display(dry.to_frame())

In [ ]:
# THE FULL CAMPAIGN. Uncomment to run - budget 45-90 minutes on a T4,
# because step 1 alone is a 200-trial Optuna study.
#
# result = run_campaign("configs/campaigns/example_extra_tuning.yaml",
#                       cfg=cfg, tracker=tracker)
# display(result.to_frame())
# print("errors:", result.errors or "none")
# print("written:", result.artefacts)

print("Full campaign left commented out so this notebook stays runnable end to end.")
print("Run it from the shell instead, where a disconnect does not lose the cell:")
print("   make campaign CAMPAIGN=configs/campaigns/example_extra_tuning.yaml")

### 5.2 · You writing a new campaign — the actual demonstration

This is the part the brief asks to see: **you** launching a new tuning campaign
by editing YAML only.

The question below is one an examiner might genuinely ask in the viva — *"what
if the cost assumptions were different?"* — and answering it takes a YAML file
and no code at all.

In [ ]:
import tempfile, yaml
from pathlib import Path

# A brand-new campaign, written here, in YAML, with nothing in src/ changed.
new_campaign = {
    "name": "cost_sensitivity_sweep",
    "description": "How does the fraud operating point move as the cost of a missed fraud changes?",
    "author": "Dipesh Kumar Yadav",
    "steps": [
        {
            "module": "transactions",
            "operation": "threshold",
            "models": ["lightgbm"],
            "note": f"Sensitivity of the operating point to a missed-fraud cost of {cost:,} INR.",
            "params": {"cost_false_negative": cost, "cost_false_positive": 500},
        }
        for cost in (5_000, 10_000, 25_000, 50_000)
    ],
}

campaign_path = Path("configs/campaigns/cost_sensitivity_sweep.yaml")
campaign_path.write_text(yaml.safe_dump(new_campaign, sort_keys=False), encoding="utf-8")
print(campaign_path.read_text()[:700])

In [ ]:
# Run it. Threshold optimisation RETRAINS NOTHING - the threshold is a
# decision parameter, not a model parameter - so a four-point cost sweep is
# seconds, not hours.
sweep = run_campaign(campaign_path, cfg=cfg, tracker=tracker)
print("ok:", sweep.ok, "| errors:", sweep.errors or "none")

frame = sweep.to_frame()
display(frame[["cost_false_negative", "threshold", "n_flagged",
               "precision", "recall", "total_cost", "saving_vs_best_baseline"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(frame["cost_false_negative"], frame["threshold"], marker="o", lw=2, color=color("teal"))
axes[0].set_xlabel("Cost of a missed fraud (INR)"); axes[0].set_ylabel("Optimal threshold")
axes[0].set_title("A costlier miss lowers the threshold")

axes[1].plot(frame["cost_false_negative"], frame["recall"], marker="o", lw=2,
             color=color("navy"), label="recall")
axes[1].plot(frame["cost_false_negative"], frame["precision"], marker="s", lw=2,
             color=semantic_color("warning"), label="precision")
axes[1].set_xlabel("Cost of a missed fraud (INR)"); axes[1].set_ylabel("Rate")
axes[1].set_title("...so the desk investigates more and catches more"); axes[1].legend()
plt.tight_layout()
save_figure(fig, "05_cost_sensitivity_sweep", close=False); plt.show()

print("\nThat entire analysis was a YAML file. No code in src/ was modified.")
print("THAT is the Level-4 hook.")

## 6 · Persist

In [ ]:
cfg.paths.tables.mkdir(parents=True, exist_ok=True)

staged.to_frame().to_csv(cfg.paths.tables / "05_staged_boosting_loans.csv", index=False)
ladder.to_csv(cfg.paths.tables / "05_finetuning_ladder_loans.csv", index=False)
if TORCH_AVAILABLE:
    lora_table.to_csv(cfg.paths.tables / "05_lora_parameter_budget.csv", index=False)
    history.to_csv(cfg.paths.tables / "05_pretraining_history.csv", index=False)
schedule_df.to_csv(cfg.paths.tables / "05_unfreeze_schedule.csv", index=False)
rates_df.to_csv(cfg.paths.tables / "05_discriminative_lrs.csv", index=False)

print("Tables written:")
for path in sorted(cfg.paths.tables.glob("05_*.csv")):
    print("   ", path.name)
print("\nCampaign results:")
for path in sorted(cfg.paths.tables.glob("campaign_*.csv")):
    print("   ", path.name)

---

## Phase 6 summary

| Level | Delivered | Evidence |
|---|---|---|
| **3a** | staged/warm-start boosting via `init_model` | per-stage trace with early stopping |
| **3b** | self-supervised FT-Transformer + LoRA per module | reconstruction-loss curve; **parameter-budget table** |
| **3c** | gradual unfreezing + discriminative LRs | schedule table; rate ladder |
| **4** | YAML-only campaign hook | worked example + **a new campaign written and run in §5.2** |

### The honest framings to carry into the viva

* **LoRA is applied to a tabular backbone, not a language model**, because no
  text data exists here. Adapting an LLM would have been decorative.
* **A higher level need not win.** If staged boosting does not beat the Level-2
  optimum, report it — complexity is explicitly *not* rewarded by the brief, and
  a negative result that is measured is worth more than a gain that is claimed.
* **Threshold optimisation retrains nothing**, which is why a cost sensitivity
  is seconds and belongs in a campaign.

**NEXT:** `06_evaluation_and_explainability` — SHAP across all modules, model
cards, calibration and PSI on the **untouched** fraud holdout (scored once,
finally), VaR backtesting with the Kupiec test, the portfolio backtest, and the
Module-11 integrated ₹1,000 crore allocation.